<a href="https://colab.research.google.com/github/JanMosz/Medikinet/blob/main/MedUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask flask-ngrok


In [ ]:
from flask import Flask, request, redirect, session
from flask import render_template_string
from flask_ngrok import run_with_ngrok
import sqlite3

app = Flask(__name__)
app.secret_key = "medica_secret"
run_with_ngrok(app)

# --- BAZA DANYCH ---

def init_db():
    conn = sqlite3.connect("medica.db")
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS patients (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            login TEXT,
            password TEXT,
            name TEXT,
            surname TEXT,
            pesel TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

# --- HTML ---

form_html = """
    <h2>Dodaj pacjenta</h2>
    <form method="POST">
        Login: <input name="login"><br>
        Hasło: <input name="password" type="password"><br>
        Imię: <input name="name"><br>
        Nazwisko: <input name="surname"><br>
        PESEL: <input name="pesel"><br>
        <button type="submit">Zapisz</button>
    </form>
    <a href="/login">Logowanie</a>
"""

login_html = """
    <h2>Logowanie</h2>
    <form method="POST">
        Login: <input name="login"><br>
        Hasło: <input name="password" type="password"><br>
        <button type="submit">Zaloguj</button>
    </form>
"""

card_html = """
    <h2>Karta pacjenta</h2>
    Imię: {{name}} <br>
    Nazwisko: {{surname}} <br>
    PESEL: {{pesel}} <br>
    <a href="/logout">Wyloguj</a>
"""

# --- ROUTES ---

@app.route("/", methods=["GET", "POST"])
def add_patient():
    if request.method == "POST":
        conn = sqlite3.connect("medica.db")
        c = conn.cursor()
        c.execute("INSERT INTO patients (login, password, name, surname, pesel) VALUES (?, ?, ?, ?, ?)",
                  (request.form["login"], request.form["password"],
                   request.form["name"], request.form["surname"],
                   request.form["pesel"]))
        conn.commit()
        conn.close()
        return redirect("/login")
    return render_template_string(form_html)

@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        conn = sqlite3.connect("medica.db")
        c = conn.cursor()
        c.execute("SELECT * FROM patients WHERE login=? AND password=?",
                  (request.form["login"], request.form["password"]))
        user = c.fetchone()
        conn.close()

        if user:
            session["user_id"] = user[0]
            return redirect("/card")
        else:
            return "Błędne dane"

    return render_template_string(login_html)

@app.route("/card")
def card():
    if "user_id" not in session:
        return redirect("/login")

    conn = sqlite3.connect("medica.db")
    c = conn.cursor()
    c.execute("SELECT name, surname, pesel FROM patients WHERE id=?",
              (session["user_id"],))
    user = c.fetchone()
    conn.close()

    return render_template_string(card_html,
                                  name=user[0],
                                  surname=user[1],
                                  pesel=user[2])

@app.route("/logout")
def logout():
    session.clear()
    return redirect("/login")

app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
Exception in thread Thread-3:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
            